In [ ]:
import pathlib

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import SVG

from vizopt.animation import SnapshotCallback
from vizopt.base import OptimConfig
from vizopt.components import stars
from vizopt.examples.sets import (
    make_animals_graph,
    make_multiples_of_primes_graph,
    make_simple_three_element_graph,
)
from vizopt.schedules import make_term_schedules
from vizopt.templates.euler.stars_vs_circles import EulerDiagram

In [ ]:
fig_folder = pathlib.Path().resolve().parents[1] / "docs" / "blog" / "img"
fig_folder, fig_folder.exists()

# Simple example: A, B, C

In [ ]:
G_simple = make_simple_three_element_graph()
representation = stars.BSpline(k_angles=64)
diagram_s = EulerDiagram.from_graph(
    G_simple,
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_smoothness=3.0,
    weight_position_anchor=0.3,
    weight_circle_collision=20.0,
    weight_set_attraction=5.0,
    representation=representation,
)
diagram_s.optimize(OptimConfig(n_iters=4000, learning_rate=0.002))

_, ax = plt.subplots(figsize=(7, 4))
diagram_s.plot(ax=ax)
plt.title("Simple example: sets {A,B}, {B,C}, {A,C}")
plt.tight_layout()
plt.savefig(fig_folder / "simple_three_sets.svg")

# Sets of multiples

In [ ]:
primes = [2, 3, 5]
G = make_multiples_of_primes_graph(primes)
elements = sorted(n for n in G.nodes if G.out_degree(n) == 0)

In [ ]:
n_iters = 8000
best_params = {
    "collision_delay": 0.27,
    "collision_ramp":  0.33,
    "exclusion_delay": 0.32,
    "exclusion_ramp":  0.47,
    "area_delay":      0.31,
    "area_ramp":       0.27,
    "perimeter_delay": 0.69,
    "perimeter_ramp":  0.45,
    "attraction_peak": 0.69,
    "attraction_ramp": 0.18,
}
term_schedules = make_term_schedules(best_params, n_iters)

In [ ]:
snapshot_cb = SnapshotCallback(every=50)

diagram = EulerDiagram.from_graph(
    G,
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.3,
    weight_circle_collision=20.0,
    weight_set_attraction=5.0,
    term_schedules=term_schedules,
)
diagram.optimize(OptimConfig(n_iters=n_iters, learning_rate=0.002), callback=snapshot_cb)
print(f"Captured {len(snapshot_cb.snapshots)} snapshots")

In [ ]:
diagram.plot()
plt.title("Multiples of 2, 3, 5 among 1-11")
plt.show()

In [ ]:
pd.DataFrame(diagram.result_.history).set_index("iteration").plot(marker=".", figsize=(7, 3))
plt.ylabel("Loss value")
plt.title("Optimization history")
plt.tight_layout()

In [ ]:
svg = diagram.animate_svg(snapshot_cb, fps=5, size=600)
SVG(data=svg)

# Animals taxonomy

In [ ]:
G_a = make_animals_graph()
elements_a = sorted(n for n in G_a.nodes if G_a.out_degree(n) == 0)
set_names_a = [n for n in nx.topological_sort(G_a) if G_a.out_degree(n) > 0]

In [ ]:
diagram_a = EulerDiagram.from_graph(
    G_a,
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.01,
    weight_circle_collision=100.0,
    weight_set_attraction=0.1,
    circle_collision_alpha=1.0,
    term_schedules=term_schedules,
)
diagram_a.optimize(OptimConfig(n_iters=n_iters, learning_rate=0.002))

In [ ]:
diagram_a.result_.history[-1]

In [ ]:
diagram_a.plot()
plt.title("Animal taxonomy")
plt.show()

# Label rectangles

Re-run the animals example with `label_rect_size` to reserve a floating label area at the top of each set boundary.

In [ ]:
snapshot_cb = SnapshotCallback(every=50)
representation = stars.BSpline(n_ctrl_pts=16, k_angles=128)
n_iters = 12000
diagram_l = EulerDiagram.from_graph(
    G_a,
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.01,
    weight_circle_collision=100.0,
    weight_set_attraction=0.1,
    circle_collision_alpha=1.0,
    label_rect_size=(0.4, 0.1),
    weight_bounding_box=1.0,
    weight_label_enclosure=20.0,
    weight_label_element_exclusion=20.0,
    weight_label_collision=10.0,
    weight_label_top=1.0,
    term_schedules=term_schedules,
    representation=representation,
)
diagram_l.optimize(OptimConfig(n_iters=n_iters, learning_rate=0.003), callback=snapshot_cb)

In [ ]:
diagram_l.plot()
plt.title("An Euler diagram representing a very limited animal taxonomy")
plt.savefig(fig_folder / "euler_animal_taxonomy.svg")

In [ ]:
svg = diagram_l.animate_svg(snapshot_cb, fps=50, size=600)
SVG(data=svg)

# Illustrating objective terms: area, perimeter, smoothness, convexity

Five circles are arranged in a C-shape (opening to the right). We run four separate
optimisations, each focusing on one term (weight = 20) while the others stay low (weight = 2).
Circle positions are strongly anchored so that only the boundary shape changes.

| Term | Effect |
|------|--------|
| **Area** | Minimises enclosed area → tight boundary, may pull inward where there are no circles |
| **Perimeter** | Minimises boundary length → prefers round/circular shapes |
| **Smoothness** | Penalises abrupt radius changes between adjacent angles → boundary follows circles but without spikes |
| **Convexity** | Penalises concave turns → explicitly forces a convex boundary |

In [ ]:
# 5 circles on a C-arc (opening to the right): angles 60°, 120°, 180°, 240°, 300°
arc_angles_c = np.array([np.pi / 3, 2 * np.pi / 3, np.pi, 4 * np.pi / 3, 5 * np.pi / 3])
R_arc = 2.5

G_terms = nx.DiGraph()
for i, a in enumerate(arc_angles_c):
    G_terms.add_node(f"c{i}", center=[float(R_arc * np.cos(a)), float(R_arc * np.sin(a))], r=0.4)
G_terms.add_node("S")
for i in range(5):
    G_terms.add_edge("S", f"c{i}")

leaf_names_terms = [n for n in G_terms.nodes if G_terms.out_degree(n) == 0]

In [ ]:
import matplotlib
cmap = matplotlib.cm.get_cmap("Set1")

_kw = dict(
    weight_enclosure=1000.,
    weight_exclusion=0.0,
    weight_position_anchor=50.0,
    weight_circle_collision=0.0,
    weight_set_attraction=0.0,
    circle_collision_alpha=0.0,
)
_optim = OptimConfig(n_iters=10000, learning_rate=0.003)

_cases_def = [
    ("Area",       cmap(0),      dict(weight_area=10.0, weight_perimeter=1.0,  weight_smoothness=0.5, weight_convexity=0.0)),
    ("Perimeter",  cmap(1),         dict(weight_area=1.0,  weight_perimeter=10.0, weight_smoothness=0.5, weight_convexity=0.0)),
    ("Smoothness", cmap(2),   dict(weight_area=1.0,  weight_perimeter=1.0,  weight_smoothness=10.0, weight_convexity=0.0)),
    #("Convexity",  cmap(3), dict(weight_area=1.0,  weight_perimeter=1.0,  weight_smoothness=0.5, weight_convexity=20.0, convexity_alpha=0.1)),
]

cases = []
for _name, _color, _params in _cases_def:
    _d = EulerDiagram.from_graph(G_terms, **_kw, **_params, offsets=0.2)
    _d.optimize(_optim)
    cases.append((_name, _color, _params, dict(zip(_d.set_names, _d.sets_)), _d.result_.history[-1]))

In [ ]:
import matplotlib
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from pandas.api.types import is_numeric_dtype

def scale_categorical_data(cat_data):
    """Transform categorical data to values between 0 and 1"""
    for col in cat_data.columns:
        cat_values = get_categorical_values(cat_data[col])
        value_map = dict(zip(cat_values, np.arange(len(cat_values))))
        cat_data[col] = cat_data[col].map(value_map) / (len(cat_values) - 1)
    return cat_data

def get_categorical_values(cat_series):
    """Get unique values in a categorical column"""
    return sorted(cat_series.astype(str).unique())

def scale_data(data, method="min_max"):
    """Scale data

    Numeric columns and categorical columns are considered separately.

    Args:
        data: a pandas DataFrame
        method: one of 'min_max', 'quantile' or 'none'
    """
    num_columns = numeric_columns(data)
    other_columns = [col for col in data.columns if col not in num_columns]
    num_data = data.loc[:, num_columns]
    other_data = data.loc[:, other_columns]
    if method == "min_max":
        low, high = min_max_scale(num_data)
    elif method == "quantile":
        low, high = quantile_scale(num_data)
    elif method == "none":
        low = pd.Series(0, index=num_data.columns)
        high = pd.Series(1, index=num_data.columns)
    else:
        raise ValueError(f"Unknown scaling method: {method}")
    num_data = (num_data - low) / (high - low)
    other_data = scale_categorical_data(other_data)
    return pd.concat([num_data, other_data], axis=1), low, high


def numeric_columns(data):
    """Get a list of numeric columns"""
    num_columns = [col for col in data.columns if is_numeric_dtype(data[col].dtypes)]
    return num_columns


def min_max_scale(num_data):
    """Get low and high values for min-max scaling"""
    high = num_data.max()
    low = num_data.min()
    return low, high

def radar_chart(
    data: pd.DataFrame,
    color_column: str | None = None,
    cmap=None,
    plot_kwargs=None,
    ax=None,
    scale_method="min_max",
    r_min=0.2,
):
    """Radar chart (spider/star chart) — parallel coordinates on polar axes.

    Each dimension is mapped to an evenly spaced angle around the circle.
    The polygon for every row is closed so the first axis connects back
    to the last.

    Args:
        data: a pandas DataFrame
        color_column: a column to color by
        cmap: a colormap or name of a colormap
        plot_kwargs: kwargs to pass to plt.plot
        ax: an optional *polar* matplotlib axis
        scale_method: method to scale the data, one of 'min_max', 'none' or 'quantile'
        r_min: radius that corresponds to a scaled value of 0 (default 0.2).
            A scaled value of 1 maps to radius 1. Set to 0 for no inner margin.

    Returns:
        the polar axis
    """
    if plot_kwargs is None:
        plot_kwargs = {"alpha": 0.5}

    scaled_data, _, _ = scale_data(data, method=scale_method)
    if color_column is not None and color_column in scaled_data.columns:
        scaled_data = scaled_data.drop(columns=[color_column])

    # remap [0, 1] → [r_min, 1] so the centre is not at zero
    scaled_data = r_min + scaled_data * (1 - r_min)

    n_dims = len(scaled_data.columns)
    angles = np.linspace(0, 2 * np.pi, n_dims, endpoint=False).tolist()
    # close the polygon
    angles += angles[:1]

    if ax is None:
        _, ax = plt.subplots(subplot_kw={"projection": "polar"})

    # determine colours
    if color_column is None or color_column not in data.columns:
        color_scheme = "constant"
    elif is_numeric_dtype(data[color_column]):
        color_scheme = "linear"
    else:
        color_scheme = "categorical"

    if isinstance(cmap, str):
        cmap = plt.get_cmap(cmap)

    if color_scheme == "categorical":
        class_names = data[color_column].unique()
        if cmap is None:
            cmap = matplotlib.cm.get_cmap("Set1")
        for i_class, class_name in enumerate(class_names):
            rows = scaled_data[data[color_column] == class_name]
            for _, row in rows.iterrows():
                values = row.values.tolist() + row.values[:1].tolist()
                ax.plot(angles, values, color=cmap(i_class), **plot_kwargs)
    elif color_scheme == "linear":
        if cmap is None:
            cmap = matplotlib.cm.get_cmap("viridis")
        vmin = data[color_column].min()
        vmax = data[color_column].max()
        color_norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        colors = cmap(color_norm(data[color_column].values))
        for i_row in range(scaled_data.shape[0]):
            values = scaled_data.iloc[i_row].values.tolist()
            values += values[:1]
            ax.plot(angles, values, color=colors[i_row], **plot_kwargs)
    else:
        for i_row in range(scaled_data.shape[0]):
            values = scaled_data.iloc[i_row].values.tolist()
            values += values[:1]
            ax.plot(angles, values, **plot_kwargs)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(scaled_data.columns)
    ax.set_ylim(0, 1)
    ax.set_yticks([])

    return ax

In [ ]:
from matplotlib.gridspec import GridSpec

_ABBREV = {
    "weight_area": "area", "weight_perimeter": "perim",
    "weight_smoothness": "smooth",
    "convexity_alpha": "α",
}

unweighted_terms = {}
for name, color, params, named_res, last_iter in cases:
    unweighted_terms[name] = {
        key.replace("_unweighted", ""): val
        for key, val in last_iter.items()
        if "unweighted" in key
    }
unweighted_terms = pd.DataFrame(unweighted_terms)
key_terms_df = unweighted_terms.T[["smoothness", "convexity", "area", "perimeter", "bounding_box"]]

n_cases = len(cases)
fig = plt.figure(figsize=(4 * n_cases, 7))
gs = GridSpec(2, n_cases, figure=fig, height_ratios=[1.2, 1.], hspace=0.05)

for i, (name, color, params, named_res, last_iter) in enumerate(cases):
    ax = fig.add_subplot(gs[0, i])
    parts = [f"{_ABBREV[k]}={v:g}" for k, v in params.items() if k in _ABBREV]
    ax.set_title(f"{name}\n({', '.join(parts)})", fontsize=10, fontweight="bold", color=color)

    res = named_res["S"]
    cx, cy = res["center"]
    radii = res["radii"]
    angles = res["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.2, color=color)
    ax.plot(bx, by, color=color, linewidth=2)

    for n in leaf_names_terms:
        ccx, ccy = G_terms.nodes[n]["center"]
        r = G_terms.nodes[n]["r"]
        ax.add_patch(plt.Circle((ccx, ccy), r, facecolor="lightyellow", alpha=0.9,
                                edgecolor="dimgray", linewidth=1.5))

    ax.set_aspect("equal")
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-4.5, 4.5)
    ax.axis("off")

mid = n_cases // 2
ax_radar = fig.add_subplot(gs[1, mid], projection="polar")
ax_radar.set_prop_cycle(color=matplotlib.cm.Set1.colors)
radar_chart(key_terms_df, ax=ax_radar, plot_kwargs={"linewidth": 2})

plt.suptitle("Boundary shapes with different dominant objective terms", fontsize=11)
plt.tight_layout()

plt.savefig(fig_folder / "euler_different_objective_terms.svg")

In [ ]:
key_terms_df

## Convexity term inspection: morphing from convex to concave

A square (W = 1.5) is deformed in two directions from a baseline:

- **t > 0** — lerp toward a circle (genuinely convex at every t)
- **t = 0** — baseline square
- **t < 0** — 4-fold harmonic perturbation `r_rect + t × 0.4W × cos(4θ)`: sides indent, corners protrude → concave

Two penalty variants are shown:
- **α = 0** (pure quadratic) `sum(max(0, −sin α)²)` — near-zero gradient for small concavities
- **α = 1** (quadratic + linear) `sum(max(0, −sin α)² + max(0, −sin α))` — constant gradient even for mild concavities

In [ ]:
import jax.numpy as jnp
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

from vizopt.components.stars import (
    _multi_term_area,
    _multi_term_convexity,
    _multi_term_perimeter,
    _multi_term_smoothness,
)

K = 64
W, H = 1.5, 1.5
angles_k = np.linspace(0, 2 * np.pi, K, endpoint=False).astype(np.float32)

# Square baseline radii: r(θ) = min(W/|cosθ|, H/|sinθ|)
r_rect = np.minimum(
    W / np.abs(np.cos(angles_k)).clip(1e-9),
    H / np.abs(np.sin(angles_k)).clip(1e-9),
).astype(np.float32)

# Circle radius for the convex direction
r_circle = float(np.mean(r_rect))

# 4-fold perturbation for the concave direction (cos 4θ peaks at mid-sides)
perturb = (0.4 * W * np.cos(4 * angles_k)).astype(np.float32)


def radii_at(t: float) -> np.ndarray:
    if t >= 0:
        return (1.0 - t) * r_rect + t * r_circle
    else:
        return np.maximum(r_rect + t * perturb, 0.02)


def losses_at(t: float) -> dict:
    r = radii_at(t)
    ov = {"centers": jnp.zeros((1, 2)), "radii": jnp.array(r[None, :])}
    ip0 = {"angles": jnp.array(angles_k), "convexity_alpha": jnp.float32(0.0)}
    ip1 = {"angles": jnp.array(angles_k), "convexity_alpha": jnp.float32(1.0)}
    return {
        "convexity_q":  float(_multi_term_convexity(ov, ip0)),
        "convexity_ql": float(_multi_term_convexity(ov, ip1)),
        "smoothness":   float(_multi_term_smoothness(ov, ip0)),
        "area":         float(_multi_term_area(ov, ip0)),
        "perimeter":    float(_multi_term_perimeter(ov, ip0)),
    }


t_dense = np.linspace(-1.0, 1.0, 400)
t_show  = np.linspace(-1.0, 1.0, 7)
all_losses = [losses_at(t) for t in t_dense]

# ── figure ────────────────────────────────────────────────────────────────────
n = len(t_show)
fig = plt.figure(figsize=(16, 8))
outer = GridSpec(2, 1, figure=fig, height_ratios=[1.2, 1.2], hspace=0.4)
gs_top = GridSpecFromSubplotSpec(1, n, subplot_spec=outer[0], wspace=0.05)
gs_bot = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[1], wspace=0.35)

# shape thumbnails
for i, t in enumerate(t_show):
    ax = fig.add_subplot(gs_top[0, i])
    r = radii_at(t)
    bx = np.append(r * np.cos(angles_k), r[0] * np.cos(angles_k[0]))
    by = np.append(r * np.sin(angles_k), r[0] * np.sin(angles_k[0]))
    color = plt.cm.RdYlGn((t + 1) / 2)
    ax.fill(bx, by, alpha=0.3, color=color)
    ax.plot(bx, by, color=color, linewidth=1.5)
    ax.set_xlim(-2.8, 2.8)
    ax.set_ylim(-2.0, 2.0)
    ax.set_aspect("equal")
    ax.set_title(f"t = {t:+.2f}", fontsize=9)
    ax.axis("off")

# convexity: quadratic vs quadratic+linear
ax_c = fig.add_subplot(gs_bot[0, 0])
ax_c.plot(t_dense, [l["convexity_q"]  for l in all_losses],
          color="mediumseagreen", linewidth=2, label="α = 0  (quadratic)")
ax_c.plot(t_dense, [l["convexity_ql"] for l in all_losses],
          color="mediumseagreen", linewidth=2, linestyle="--", label="α = 1  (quadratic + linear)")
ax_c.axvspan(-1.0, 0.0, alpha=0.07, color="red")
ax_c.axvspan(0.0,  1.0, alpha=0.07, color="green")
ax_c.axvline(0, color="black", linewidth=0.8, alpha=0.4)
for t in t_show:
    ax_c.axvline(t, color="gray", linestyle="--", linewidth=0.7, alpha=0.4)
ax_c.set_xlabel("t")
ax_c.set_ylabel("Loss value")
ax_c.set_title("Convexity term")
ax_c.legend(fontsize=9)

# other terms for comparison
ax_o = fig.add_subplot(gs_bot[0, 1])
for name, col in [("smoothness", "mediumpurple"), ("area", "steelblue"), ("perimeter", "tomato")]:
    ax_o.plot(t_dense, [l[name] for l in all_losses], label=name, color=col, linewidth=2)
ax_o.axvspan(-1.0, 0.0, alpha=0.07, color="red")
ax_o.axvspan(0.0,  1.0, alpha=0.07, color="green")
ax_o.axvline(0, color="black", linewidth=0.8, alpha=0.4)
for t in t_show:
    ax_o.axvline(t, color="gray", linestyle="--", linewidth=0.7, alpha=0.4)
ax_o.set_xlabel("t")
ax_o.legend()
ax_o.set_title("Other terms")

plt.suptitle(
    "Objective terms along the shape morph\nconcave (t < 0)  ←——  square (t = 0)  ——→  circle (t > 0)",
    fontsize=11,
)
plt.tight_layout()